# Signal Lab - Label Learnability Workbench

This notebook replaces the old parameter-only Signal Lab workflow.

It now does both jobs in one place:

1. design and compare regime-aware label profiles
2. run lightweight learnability diagnostics before deep-learning training

Production label and feature logic still comes from `Learn/`. Notebook-specific orchestration lives in `Lables/`.


In [8]:
from copy import deepcopy
from datetime import datetime
from pathlib import Path
import importlib
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import Markdown, display

import Lables.baselines as baselines_module
import Lables.config as config_module
import Lables.data as data_module
import Lables.features as features_module
import Lables.labels as labels_module
import Lables.mutual_information as mutual_information_module
import Lables.nearest_neighbors as nearest_neighbors_module
import Lables.regime_analysis as regime_analysis_module
import Lables.scoring as scoring_module
import Lables.separability as separability_module
import Lables.visualisation as visualisation_module

for _module in [
    baselines_module,
    config_module,
    data_module,
    features_module,
    labels_module,
    mutual_information_module,
    nearest_neighbors_module,
    regime_analysis_module,
    scoring_module,
    separability_module,
    visualisation_module,
]:
    importlib.reload(_module)

evaluate_baseline_models = baselines_module.evaluate_baseline_models
build_experiment_config = config_module.build_experiment_config
load_label_profiles = data_module.load_label_profiles
load_ohlcv_dataset = data_module.load_ohlcv_dataset
build_research_features = features_module.build_research_features
events_to_bar_labels = labels_module.events_to_bar_labels
generate_label_events = labels_module.generate_label_events
summarise_label_quality = labels_module.summarise_label_quality
evaluate_mutual_information = mutual_information_module.evaluate_mutual_information
evaluate_neighbor_consistency = nearest_neighbors_module.evaluate_neighbor_consistency
evaluate_regime_stability = regime_analysis_module.evaluate_regime_stability
aggregate_scores = scoring_module.aggregate_scores
rank_label_profiles = scoring_module.rank_label_profiles
evaluate_class_separability = separability_module.evaluate_class_separability
plot_feature_importance = visualisation_module.plot_feature_importance
plot_label_distribution = visualisation_module.plot_label_distribution
plot_projection = visualisation_module.plot_projection
plot_regime_performance = visualisation_module.plot_regime_performance
plot_signal_chart = visualisation_module.plot_signal_chart

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


## Configuration

Adjust the symbol, label profiles, row budget, and optional inline overrides here.


In [ ]:
CONFIG = build_experiment_config(
    symbol="EURUSD",
    label_profiles=["EURUSD_1m_dev", "EURUSD_1m_r13"],
    n_rows=250_000,
    feature_builder="_add_features_light",
    include_mtf=False,
    baseline_models=("logistic", "random_forest", "lightgbm"),
    save_outputs=False,
)

PROFILE_OVERRIDES = {
    # "EURUSD_1m_dev": {
    #     "label_params": {"trend_pullback_thresh": 0.70}
    # }
}

SELECTED_PROFILE_FOR_PLOTS = None

config_preview = pd.DataFrame(
    {
        "setting": list(CONFIG.keys()),
        "value": [CONFIG[key] for key in CONFIG.keys()],
    }
)
display(config_preview)


,setting,value
0,symbol,EURUSD
1,label_profiles,"[EURUSD_1m_dev, EURUSD_1m_r13]"
2,dataset_path,C:\Users\Stuart\Desktop\Trading Bot\MLQ5-Produ...
3,feature_builder,auto
4,n_rows,250000
5,include_mtf,False
6,temporal_folds,3
7,baseline_models,"[logistic, random_forest, lightgbm]"
8,baseline_train_rows,100000
9,baseline_test_rows,50000


## Load dataset and label profiles

This keeps the current repo contract: OHLCV comes from `../data/...` and profile definitions come from `params/label_params.json`.


In [10]:
def apply_profile_overrides(profile_map, overrides):
    merged = deepcopy(profile_map)
    for profile_name, sections in overrides.items():
        if profile_name not in merged:
            raise KeyError(f"Unknown profile override target: {profile_name}")
        for section_name, section_values in sections.items():
            if section_name not in merged[profile_name]:
                raise KeyError(f"Unknown section {section_name!r} for profile {profile_name!r}")
            merged[profile_name][section_name].update(section_values)
    return merged


df_raw = load_ohlcv_dataset(CONFIG["dataset_path"], n_rows=CONFIG["n_rows"])
profiles = apply_profile_overrides(load_label_profiles(CONFIG["label_profiles"]), PROFILE_OVERRIDES)

dataset_summary = pd.DataFrame(
    [
        {
            "rows": len(df_raw),
            "start": df_raw["Time"].min(),
            "end": df_raw["Time"].max(),
            "symbol": CONFIG["symbol"],
            "dataset_path": CONFIG["dataset_path"],
        }
    ]
)
profile_preview = pd.DataFrame(
    [
        {
            "profile": profile_name,
            "comment": profile.get("comment", ""),
            **profile["regime_params"],
            **profile["label_params"],
        }
        for profile_name, profile in profiles.items()
    ]
)

display(dataset_summary)
display(profile_preview)
display(df_raw.head())


,rows,start,end,symbol,dataset_path
0,250000,2025-09-18 11:36:00+00:00,2026-05-22 20:54:00+00:00,EURUSD,C:\Users\Stuart\Desktop\Trading Bot\MLQ5-Produ...


,profile,comment,ma_period,slope_smoothness,regime_min_duration,atr_window,atr_lookback,atr_percentile,slope_threshold,z_window,z_thresh,z_limit,tp_mult,sl_mult,max_horizon,trend_pullback_thresh,skip_range
0,EURUSD_1m_dev,EURUSD 1m dev profile. Tuned over R6-R10: redu...,60,50,0,14,720,0.0,0.000003,14,1,5,2.6,2.0,90,0.85,True
1,EURUSD_1m_r13,From EURUSD_1m_dev. Labeller updated to use H/...,60,50,0,14,720,0.0,0.000003,14,1,5,2.6,2.0,90,0.85,True


,Time,Open,High,Low,Close,Volume
0,2025-09-18 11:36:00+00:00,1.18323,1.18330,1.18303,1.18309,173
1,2025-09-18 11:37:00+00:00,1.18308,1.18312,1.18284,1.18294,159
2,2025-09-18 11:38:00+00:00,1.18293,1.18294,1.18257,1.18263,167
3,2025-09-18 11:39:00+00:00,1.18262,1.18270,1.18257,1.18267,182
4,2025-09-18 11:40:00+00:00,1.18268,1.18276,1.18255,1.18260,186


## Run label and learnability analysis

Each profile reuses the existing production labeler and feature stack, then runs a lightweight research pass over the resulting bar labels.


In [ ]:
profile_results = {}

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for profile_name, profile in tqdm(
        profiles.items(),
        total=len(profiles),
        desc="Running learnability analysis",
    ):
        regime_params = deepcopy(profile["regime_params"])
        label_params = deepcopy(profile["label_params"])

        events = generate_label_events(df_raw, regime_params, label_params)
        bar_labels = events_to_bar_labels(df_raw, events, rollover_window=CONFIG["rollover_window"])
        label_summary = summarise_label_quality(
            df_raw,
            regime_params,
            label_params,
            events=events,
            rollover_window=CONFIG["rollover_window"],
        )

        X, y, feature_names, feature_frame = build_research_features(
            df_raw,
            bar_labels,
            symbol=CONFIG["symbol"],
            feature_builder=CONFIG["feature_builder"],
            regime_params=regime_params,
            include_mtf=CONFIG["include_mtf"],
            max_rows=CONFIG["n_rows"],
        )

        baseline = evaluate_baseline_models(
            X,
            y,
            model_names=CONFIG["baseline_models"],
            n_splits=CONFIG["temporal_folds"],
            max_train_rows=CONFIG["baseline_train_rows"],
            max_test_rows=CONFIG["baseline_test_rows"],
        )
        mutual_information = evaluate_mutual_information(
            X,
            y,
            feature_names=feature_names,
            max_samples=CONFIG["mi_rows"],
        )
        separability = evaluate_class_separability(
            X,
            y,
            max_samples=CONFIG["projection_rows"],
        )
        nearest_neighbors = evaluate_neighbor_consistency(
            X,
            y,
            n_neighbors=CONFIG["neighbor_k"],
            max_train_rows=CONFIG["neighbor_train_rows"],
            max_eval_rows=CONFIG["neighbor_eval_rows"],
        )

        if "Regime" in feature_frame.columns:
            regime_values = feature_frame["Regime"].to_numpy()
        else:
            regime_values = np.zeros(len(feature_frame), dtype=int)

        regime_analysis = evaluate_regime_stability(
            X,
            y,
            regimes=regime_values,
            max_test_rows=CONFIG["regime_test_rows"],
        )

        aggregate = aggregate_scores(
            {
                "baseline": baseline,
                "mutual_information": mutual_information,
                "separability": separability,
                "nearest_neighbors": nearest_neighbors,
                "regime_analysis": regime_analysis,
            },
            weights=CONFIG["score_weights"],
        )

        profile_results[profile_name] = {
            "config": profile,
            "events": events,
            "bar_labels": bar_labels,
            "feature_names": feature_names,
            "feature_frame": feature_frame,
            "label_summary": label_summary,
            "baseline": baseline,
            "mutual_information": mutual_information,
            "separability": separability,
            "nearest_neighbors": nearest_neighbors,
            "regime_analysis": regime_analysis,
            "aggregate": aggregate,
        }

label_summary_df = pd.DataFrame(
    [
        {"label_profile": profile_name, **result["label_summary"]}
        for profile_name, result in profile_results.items()
    ]
).sort_values("label_profile").reset_index(drop=True)

learnability_df = rank_label_profiles(profile_results, weights=CONFIG["score_weights"])
trade_view_columns = [
    "label_profile",
    "learnability_score",
    "trade_precision",
    "trade_macro_f1",
    "trade_ovr_auc",
    "directional_accuracy_on_trades",
    "sell_precision",
    "sell_recall",
    "buy_precision",
    "buy_recall",
    "predicted_trade_rate",
    "false_trade_rate",
    "label_density",
    "tp_rate",
]
trade_view_df = learnability_df.reindex(columns=trade_view_columns)

display(label_summary_df)
display(trade_view_df)


Running learnability analysis:   0%|          | 0/2 [00:00<?, ?it/s]

KeyError: "['trade_precision', 'trade_macro_f1', 'trade_ovr_auc', 'directional_accuracy_on_trades', 'sell_precision', 'sell_recall', 'buy_precision', 'buy_recall', 'predicted_trade_rate', 'false_trade_rate'] not in index"

## Visual diagnostics

By default the notebook plots the highest-ranked profile unless `SELECTED_PROFILE_FOR_PLOTS` is set.


In [ ]:
if learnability_df.empty:
    raise RuntimeError("No profile results were generated.")

selected_profile = SELECTED_PROFILE_FOR_PLOTS or learnability_df.iloc[0]["label_profile"]
selected = profile_results[selected_profile]

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
plot_signal_chart(df_raw, selected["events"], ax=axes[0, 0])
plot_label_distribution(selected["bar_labels"], ax=axes[0, 1])
plot_feature_importance(selected["mutual_information"], ax=axes[1, 0])
plot_projection(selected["separability"]["projection_frame"], ax=axes[1, 1])
plot_regime_performance(selected["regime_analysis"]["regime_breakdown"], ax=axes[2, 0])
axes[2, 1].axis("off")
axes[2, 1].text(
    0.0,
    1.0,
    "Selected profile: {}\nLearnability score: {:.3f}\nTrade precision: {:.3f}\nTrade macro F1: {:.3f}\nDirectional accuracy on trades: {:.3f}".format(
        selected_profile,
        selected["aggregate"]["learnability_score"],
        selected["baseline"].get("mean_trade_precision", float("nan")),
        selected["baseline"].get("mean_trade_macro_f1", float("nan")),
        selected["baseline"].get("mean_directional_accuracy_on_trades", float("nan")),
    ),
    va="top",
    fontsize=11,
)
plt.tight_layout()
plt.show()

display(selected["regime_analysis"]["regime_breakdown"])


## Optional save step

Set `CONFIG["save_outputs"] = True` to persist the ranking tables for later review.


In [ ]:
if CONFIG["save_outputs"]:
    output_dir = Path(CONFIG["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

    label_path = output_dir / f"{CONFIG['symbol']}_label_summary_{timestamp}.csv"
    learnability_path = output_dir / f"{CONFIG['symbol']}_learnability_{timestamp}.csv"
    note_path = output_dir / f"{CONFIG['symbol']}_recommendation_{timestamp}.json"

    label_summary_df.to_csv(label_path, index=False)
    learnability_df.to_csv(learnability_path, index=False)

    best_row = learnability_df.iloc[0].to_dict()
    note_payload = {
        "symbol": CONFIG["symbol"],
        "saved_at_utc": timestamp,
        "best_profile": best_row["label_profile"],
        "best_profile_scores": best_row,
    }
    note_path.write_text(json.dumps(note_payload, indent=2, default=str), encoding="utf-8")

    print(f"Saved: {label_path}")
    print(f"Saved: {learnability_path}")
    print(f"Saved: {note_path}")
else:
    print("Output saving is disabled.")


## Final recommendation

This cell turns the ranking table into a compact decision note for the next training step.


In [ ]:
best = learnability_df.iloc[0]
best_profile_name = best["label_profile"]
best_label = label_summary_df[label_summary_df["label_profile"] == best_profile_name].iloc[0]

recommendation = f"""
### Recommendation

- **Best profile:** `{best_profile_name}`
- **Learnability score:** `{best['learnability_score']:.3f}`
- **Trade precision:** `{best['trade_precision']:.3f}`
- **Trade macro F1:** `{best['trade_macro_f1']:.3f}`
- **Trade one-vs-rest AUC:** `{best['trade_ovr_auc']:.3f}`
- **Directional accuracy on trades:** `{best['directional_accuracy_on_trades']:.3f}`
- **SELL precision / recall:** `{best['sell_precision']:.3f}` / `{best['sell_recall']:.3f}`
- **BUY precision / recall:** `{best['buy_precision']:.3f}` / `{best['buy_recall']:.3f}`
- **Predicted trade rate:** `{best['predicted_trade_rate']:.2%}`
- **False trade rate:** `{best['false_trade_rate']:.2%}`
- **Label density:** `{best_label['label_density']:.2%}`
- **TP rate:** `{best_label['tp_rate']:.2%}`

Use this profile as the leading candidate for `1_2 Feature Lab.ipynb`, sweeps, or `train_prod_model_cli.py`.
"""

display(Markdown(recommendation))
